<a href="https://colab.research.google.com/github/itwasnoteasy/rag-experiments/blob/claude%2Feager-ritchie-TC17K/rag_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!git clone https://github.com/itwasnoteasy/rag-experiments.git

Cloning into 'rag-experiments'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 18 (delta 5), reused 17 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 20.13 KiB | 20.13 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [3]:
%cd rag-experiments

/content/rag-experiments


In [4]:
!pip install -q sentence-transformers chromadb rank-bm25 transformers torch langfuse google-generativeai pandas tabulate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 71.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 562.6/562.6 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 111.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0

In [5]:
!python setup.py

  RAG Experiments — Project Setup Confirmation

Corpus loaded: 20 documents

╭─────────┬──────────────────┬─────────────────────────────────────────────────────────╮
│ ID      │ Category         │ Title                                                   │
├─────────┼──────────────────┼─────────────────────────────────────────────────────────┤
│ DI-001  │ device_insurance │ How to File a Device Insurance Claim                    │
│ DI-002  │ device_insurance │ Device Insurance Deductible Amounts by Plan Tier        │
│ DI-003  │ device_insurance │ Coverage Exclusions Under Device Insurance              │
│ DI-004  │ device_insurance │ Device Swap Process After Claim Approval                │
│ DI-005  │ device_insurance │ Water Damage Policy and What's Covered                  │
│ DI-006  │ device_insurance │ Enrolling in Device Insurance After Purchase            │
│ DI-007  │ device_insurance │ Claim Limits and Claim Frequency Policy                 │
│ DI-008  │ device_insurance │ Th

In [6]:
from rank_bm25 import BM25Okapi
from corpus import CORPUS
from queries import QUERIES

# Index the corpus
tokenized_corpus = [doc["content"].lower().split() for doc in CORPUS]
bm25 = BM25Okapi(tokenized_corpus)

# Run all 10 queries
for q in QUERIES:
    tokens = q["text"].lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = scores.argsort()[::-1][:3]
    top_docs = [CORPUS[i]["id"] for i in top_idx]
    hit = "✓" if q["expected_doc"] in top_docs else "✗"
    print(f"{hit} [{q['query_type']:12}] {q['id']} | Expected: {q['expected_doc']} | Got: {top_docs}")

✓ [exact_match ] Q01 | Expected: DI-002 | Got: ['DI-002', 'DI-003', 'DI-008']
✓ [exact_match ] Q02 | Expected: DI-004 | Got: ['DI-004', 'SMB-009', 'DI-001']
✓ [semantic    ] Q03 | Expected: DI-005 | Got: ['DI-005', 'DI-006', 'DI-007']
✗ [semantic    ] Q04 | Expected: SMB-009 | Got: ['DI-001', 'SMB-006', 'SMB-007']
✗ [ambiguous   ] Q05 | Expected: DI-006 | Got: ['SMB-008', 'SMB-006', 'SMB-010']
✗ [ambiguous   ] Q06 | Expected: SMB-004 | Got: ['DI-002', 'SMB-007', 'SMB-008']
✗ [context_dep ] Q07 | Expected: DI-002 | Got: ['SMB-004', 'DI-009', 'DI-004']
✗ [context_dep ] Q08 | Expected: SMB-002 | Got: ['DI-003', 'SMB-009', 'DI-002']
✗ [clear_intent] Q09 | Expected: DI-001 | Got: ['DI-007', 'DI-005', 'SMB-007']
✗ [clear_intent] Q10 | Expected: SMB-005 | Got: ['DI-005', 'SMB-001', 'DI-007']


In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")  # fast, good quality

# Embed all documents
doc_texts = [doc["content"] for doc in CORPUS]
doc_embeddings = model.encode(doc_texts, show_progress_bar=True)

# Run all 10 queries
for q in QUERIES:
    q_emb = model.encode(q["text"])
    sims = np.dot(doc_embeddings, q_emb) / (
        np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(q_emb)
    )
    # print (sims)
    top_idx = sims.argsort()[::-1][:3]
    top_docs = [CORPUS[i]["id"] for i in top_idx]
    hit = "✓" if q["expected_doc"] in top_docs else "✗"
    print(f"{hit} [{q['query_type']:12}] {q['id']} | Expected: {q['expected_doc']} | Got: {top_docs}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✓ [exact_match ] Q01 | Expected: DI-002 | Got: ['DI-002', 'DI-006', 'DI-003']
✓ [exact_match ] Q02 | Expected: DI-004 | Got: ['DI-004', 'DI-006', 'DI-002']
✓ [semantic    ] Q03 | Expected: DI-005 | Got: ['DI-008', 'DI-005', 'DI-001']
✓ [semantic    ] Q04 | Expected: SMB-009 | Got: ['SMB-002', 'SMB-009', 'SMB-006']
✓ [ambiguous   ] Q05 | Expected: DI-006 | Got: ['DI-006', 'DI-001', 'DI-003']
✗ [ambiguous   ] Q06 | Expected: SMB-004 | Got: ['SMB-003', 'DI-007', 'DI-006']
✓ [context_dep ] Q07 | Expected: DI-002 | Got: ['DI-002', 'DI-003', 'DI-006']
✗ [context_dep ] Q08 | Expected: SMB-002 | Got: ['DI-004', 'SMB-006', 'SMB-001']
✓ [clear_intent] Q09 | Expected: DI-001 | Got: ['DI-001', 'DI-008', 'DI-005']
✓ [clear_intent] Q10 | Expected: SMB-005 | Got: ['SMB-008', 'SMB-007', 'SMB-005']


In [8]:
# Run both and compare
results = []
for q in QUERIES:
    # BM25
    tokens = q["text"].lower().split()
    bm25_top = [CORPUS[i]["id"] for i in bm25.get_scores(tokens).argsort()[::-1][:3]]

    # Dense
    q_emb = model.encode(q["text"])
    sims = np.dot(doc_embeddings, q_emb) / (np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(q_emb))
    # Experiment with top1, top2 and top3
    dense_top = [CORPUS[i]["id"] for i in sims.argsort()[::-1][:1]]

    bm25_hit = "✓" if q["expected_doc"] in bm25_top else "✗"
    dense_hit = "✓" if q["expected_doc"] in dense_top else "✗"
    results.append([q["id"], q["query_type"], q["difficulty"], q["expected_doc"],
                    f"{bm25_hit} {bm25_top[0]}", f"{dense_hit} {dense_top[0]}"])

from tabulate import tabulate
print(tabulate(results, headers=["ID","Type","Diff","Expected","BM25 Top1","Dense Top1"], tablefmt="rounded_outline"))

╭──────┬──────────────┬────────┬────────────┬─────────────┬──────────────╮
│ ID   │ Type         │ Diff   │ Expected   │ BM25 Top1   │ Dense Top1   │
├──────┼──────────────┼────────┼────────────┼─────────────┼──────────────┤
│ Q01  │ exact_match  │ easy   │ DI-002     │ ✓ DI-002    │ ✓ DI-002     │
│ Q02  │ exact_match  │ easy   │ DI-004     │ ✓ DI-004    │ ✓ DI-004     │
│ Q03  │ semantic     │ hard   │ DI-005     │ ✓ DI-005    │ ✗ DI-008     │
│ Q04  │ semantic     │ hard   │ SMB-009    │ ✗ DI-001    │ ✗ SMB-002    │
│ Q05  │ ambiguous    │ medium │ DI-006     │ ✗ SMB-008   │ ✓ DI-006     │
│ Q06  │ ambiguous    │ medium │ SMB-004    │ ✗ DI-002    │ ✗ SMB-003    │
│ Q07  │ context_dep  │ hard   │ DI-002     │ ✗ SMB-004   │ ✓ DI-002     │
│ Q08  │ context_dep  │ hard   │ SMB-002    │ ✗ DI-003    │ ✗ DI-004     │
│ Q09  │ clear_intent │ easy   │ DI-001     │ ✗ DI-007    │ ✓ DI-001     │
│ Q10  │ clear_intent │ easy   │ SMB-005    │ ✗ DI-005    │ ✗ SMB-008    │
╰──────┴──────────────┴──

In [1]:
!git pull

fatal: not a git repository (or any of the parent directories): .git


In [9]:
!python experiment_1_retrieval.py

=== Building dense index (all-MiniLM-L6-v2) ===
Loading weights: 100% 103/103 [00:00<00:00, 5575.73it/s]
Batches: 100% 1/1 [00:02<00:00,  2.09s/it]
  Indexed 20 documents into ChromaDB.

=== Building BM25 index ===
  Indexed 20 documents.

  PER-QUERY COMPARISON: Dense | BM25 | RRF

────────────────────────────────────────────────────────────────────────────────
  Q01 [exact_match] [easy]
  Query   : What is the deductible for a premium smartphone under policy INS-POL-2024?
  Expected: DI-002

╭──────────────────────┬───────────────────────┬──────────────────────╮
│ Dense Top-3          │ BM25 Top-3            │ RRF Top-3            │
├──────────────────────┼───────────────────────┼──────────────────────┤
│ #1 DI-002 (0.7574) ✓ │ #1 DI-002 (14.1318) ✓ │ #1 DI-002 (0.0328) ✓ │
│ #2 DI-006 (0.5591)   │ #2 DI-003 (5.6920)    │ #2 DI-003 (0.0320)   │
│ #3 DI-003 (0.5404)   │ #3 DI-008 (4.6116)    │ #3 DI-006 (0.0161)   │
╰──────────────────────┴───────────────────────┴─────────────────────

In [12]:
!git pull
!python experiment_2_reranking.py

remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 3), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 7.84 KiB | 1004.00 KiB/s, done.
From https://github.com/itwasnoteasy/rag-experiments
   e65cc85..137174a  claude/eager-ritchie-TC17K -> origin/claude/eager-ritchie-TC17K
Updating e65cc85..137174a
Fast-forward
 AI Conversation.md        | 298 ++++++++++++++++++++++++++++++++++++++++++++++
 experiment_2_reranking.py |  12 +-
 2 files changed, 307 insertions(+), 3 deletions(-)
 create mode 100644 AI Conversation.md
=== Loading cross-encoder (ms-marco-MiniLM-L-6-v2) ===
Loading weights: 100% 105/105 [00:00<00:00, 4147.34it/s]
  Model loaded.

  PER-QUERY DISPLACEMENT: RRF rank → Cross-encoder rank

────────────────────────────────────────────────────────────────────────────────
  Q01 [exact_match] [easy]
  Query   : What is the deductible for a premium smartphone 